# Kandinsky V2A — Persistent Worker Pilot (Colab)

## Goal

Check the deployment pattern we actually need: load Kandinsky Full Lite once, keep one pipeline process alive, and handle three video-to-audio jobs one after another.

Use the GPU Colab assigns today (G4 is fine if it has at least 76,000 MiB available). We are **not** comparing GPU models. The notebook records the hardware so the same job can later be repeated on one AWS GPU.

## What success means

- The checkpoint loads once.
- Three sequential five-second jobs finish in the same Python process.
- We record model-load time separately from each end-to-end job time.
- We capture peak GPU/RAM use, output hashes, and the exact environment.

This is a small inference and worker test. It does not train the model or test concurrent users.


## Step 1 — Check the Colab GPU and prepare a working folder

Choose the available GPU in Colab. Full Lite needs an 80 GB-class GPU for this pilot. The cell reports the actual card and stops if the assigned GPU has less than 76,000 MiB.


In [ ]:
import os, sys, csv, json, time, shutil, hashlib, subprocess, zipfile, platform, statistics
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Audio, display

if sys.version_info < (3, 13):
    raise RuntimeError(f"Python 3.13+ is required by the pinned Kandinsky environment. Found: {sys.version}")

def run_checked(command, cwd=None, label="command", env=None):
    result = subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-2500:])
        print(result.stderr[-5000:])
        raise RuntimeError(f"{label} failed.")
    return result

if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU runtime. Select a Colab GPU and reconnect.")

gpu_probe = run_checked([
    "nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
    "--format=csv,noheader,nounits"
], label="inspect GPU")
gpu_fields = next(csv.reader([gpu_probe.stdout.strip()]))
GPU_NAME = gpu_fields[0].strip()
GPU_MEMORY_MIB = int(gpu_fields[1])
GPU_COMPUTE_CAP = gpu_fields[2].strip() if len(gpu_fields) > 2 else "unknown"
if GPU_MEMORY_MIB < 76000:
    raise RuntimeError(f"Full Lite pilot requires an 80 GB-class GPU. Found {GPU_NAME}: {GPU_MEMORY_MIB} MiB.")

if "RTX PRO 6000" in GPU_NAME.upper():
    GPU_PROFILE = "g4_rtx_pro_6000"
elif "H100" in GPU_NAME.upper():
    GPU_PROFILE = "h100"
elif "A100" in GPU_NAME.upper():
    GPU_PROFILE = "a100"
else:
    GPU_PROFILE = "other_80gb_class"

storage = [Path("/content/local-scratch"), Path("/content")]
CACHE_ROOT = max((p for p in storage if p.exists()), key=lambda p: shutil.disk_usage(p).free)
WORK = CACHE_ROOT / "kandinsky_v2a_persistent_worker"
INPUT_DIR = WORK / "input"
OUTPUT_DIR = WORK / "outputs"
RESOURCE_DIR = WORK / "resources"
for path in (INPUT_DIR, OUTPUT_DIR, RESOURCE_DIR):
    path.mkdir(parents=True, exist_ok=True)

FREE_GIB = shutil.disk_usage(CACHE_ROOT).free / (1024**3)
RAM_TOTAL_GIB = int(next(x for x in Path("/proc/meminfo").read_text().splitlines()
                         if x.startswith("MemTotal:")).split()[1]) / (1024**2)
print(f"GPU: {GPU_NAME} | profile={GPU_PROFILE} | {GPU_MEMORY_MIB} MiB | compute={GPU_COMPUTE_CAP}")
print(f"Host RAM: {RAM_TOTAL_GIB:.1f} GiB | free disk: {FREE_GIB:.1f} GiB | work: {WORK}")
if FREE_GIB < 45:
    raise RuntimeError("Free at least 45 GiB on Colab storage before downloading Full Lite.")


## Step 2 — Install the pinned Kandinsky environment

The run uses the same Kandinsky source commit as our earlier A100 pilot. Dependencies and checkpoint downloads are stored in this Colab runtime; no AWS instance or API key is needed.


In [ ]:
SOURCE_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
REPO = Path("/content/kandinsky-v2a-worker-src")
KANDINSKY_HOME = CACHE_ROOT / "kandinsky-model-cache"
os.environ["KANDINSKY_HOME"] = str(KANDINSKY_HOME)

if not (REPO / ".git").exists():
    run_checked([
        "git", "clone", "--filter=blob:none",
        "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)
    ], label="clone Kandinsky")

run_checked(["git", "fetch", "origin", SOURCE_COMMIT], cwd=REPO, label="fetch pinned source")
run_checked(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO, label="checkout pinned source")
actual_commit = run_checked(["git", "rev-parse", "HEAD"], cwd=REPO, label="read source commit").stdout.strip()
if actual_commit != SOURCE_COMMIT:
    raise RuntimeError(f"Unexpected source commit: {actual_commit}")

run_checked([sys.executable, "-m", "pip", "install", "-q", "uv"], label="install uv")
if not shutil.which("just"):
    run_checked(
        ["bash", "-lc", "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"],
        label="install just"
    )

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
if setup.returncode:
    print(setup.stdout[-2500:])
    print(setup.stderr[-5000:])
    raise RuntimeError("Kandinsky dependency setup failed.")

PY = REPO / ".venv" / "bin" / "python"
if not PY.is_file():
    raise FileNotFoundError(f"Pinned Python environment was not created: {PY}")
print("Pinned commit:", actual_commit)
print("Worker Python:", PY)


## Step 3 — Prepare the same five-second CC0 video input

We use the previously selected “Big City Life” source and its fixed five-second window. The video is silent; Kandinsky receives the frames plus the same short scene prompt on every job.


In [ ]:
SOURCE_PAGE = "https://commons.wikimedia.org/wiki/File:Big_City_Life.webm"
SOURCE_URL = "https://commons.wikimedia.org/wiki/Special:Redirect/file/Big_City_Life.webm"
EXPECTED_SOURCE_SHA256 = "2834fbc411cc59f06a63f9ac7ce9ba5b748b67242d1b36a4b18043a070a7966a"
EXPECTED_WINDOW_SHA256 = "9aa4cb5e674c7b9860f1d87ba59806f8211dcc3d4b418f7e0ab258ba4b5d49cc"
PROMPT = "Several pedestrians walking on a city pavement, with soft footsteps and distant urban movement. No music or speech."
DURATION_SECONDS = 5

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_path = INPUT_DIR / "Big_City_Life.webm"
import urllib.request
download_request = urllib.request.Request(
    SOURCE_URL,
    headers={"User-Agent": "Kandinsky-V2A-Persistent-Worker-Pilot/1.0"},
)
try:
    with urllib.request.urlopen(download_request, timeout=180) as response:
        source_path.write_bytes(response.read())
except Exception as exc:
    raise RuntimeError(f"Could not download the CC0 source: {exc}") from exc
source_hash = sha256_file(source_path)
print("Source SHA-256:", source_hash)
if source_hash != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("The downloaded source hash differs from the reviewed pilot source.")

INPUT_VIDEO = INPUT_DIR / "city_walkers_00_05_silent.mp4"
prep = run_checked([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-ss", "0", "-i", str(source_path), "-t", str(DURATION_SECONDS),
    "-map", "0:v:0", "-an",
    "-vf", "fps=24,scale='min(960,iw)':-2",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
    "-pix_fmt", "yuv420p", str(INPUT_VIDEO)
], label="prepare five-second video")
window_hash = sha256_file(INPUT_VIDEO)
probe = run_checked([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(INPUT_VIDEO)
], label="inspect video duration")
window_duration = float(probe.stdout.strip())
print(f"Window: {window_duration:.3f} sec | SHA-256: {window_hash}")
if abs(window_duration - 5.0) > 0.05:
    raise RuntimeError(f"Expected a five-second window, got {window_duration:.3f}s.")
if window_hash != EXPECTED_WINDOW_SHA256:
    print("Note: encoded bytes differ from the earlier FFmpeg run. This ZIP records the exact window used here.")
print("Prompt:", PROMPT)


## Step 4 — Download Full Lite and set the worker configuration

We use Full Lite because it was the better sounding Kandinsky option in our listening so far. We will measure one model, not spend time repeating a checkpoint comparison.


In [ ]:
CONFIG = REPO / "aws_persistent_worker_lite.yaml"
CONFIG.write_text("""checkpoint: lite
cache:
  mode: none
offload:
  strategy: none
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""", encoding="utf-8")

run_checked(["just", "download", "lite"], cwd=REPO, label="download Full Lite")
WEIGHTS_DIR = KANDINSKY_HOME / "weights"
CHECKPOINT_REVISIONS = []
for ref_path in WEIGHTS_DIR.glob("models--*/refs/main"):
    revision = ref_path.read_text(encoding="utf-8").strip()
    CHECKPOINT_REVISIONS.append({"cache_repo": ref_path.parent.parent.name, "revision": revision})

package_freeze = run_checked([str(PY), "-m", "pip", "freeze"], cwd=REPO, label="record package versions").stdout
(WORK / "environment_packages.txt").write_text(package_freeze, encoding="utf-8")
print("Checkpoint refs:", json.dumps(CHECKPOINT_REVISIONS, indent=2))
print(f"Environment package entries: {len(package_freeze.splitlines())}")
print("Configuration:", CONFIG)


## Step 5 — Run three jobs through one loaded pipeline

The driver creates the Kandinsky pipeline once, then calls it three times with the same video and prompt and seeds 42, 123, and 7. Each call saves an MP4. This is the key AWS readiness check: the worker process remains alive between jobs instead of restarting the CLI for every request.

The first call includes one-time CUDA initialization. We report it separately from the later warm jobs. All job times include V2A generation and writing the result file.


In [ ]:
DRIVER = WORK / "persistent_worker_driver.py"
REPORT = WORK / "worker_report.json"
RUN_LOG = WORK / "worker.log"
REPORT.write_text("{}", encoding="utf-8")

driver_source = r"""
import csv, hashlib, json, os, sys, time, traceback
from pathlib import Path
import torch
from kandinsky import get_pipeline

spec = json.loads(sys.argv[1])
output_dir = Path(spec["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def host_memory_mib():
    rows = Path("/proc/meminfo").read_text().splitlines()
    values = {row.split(":", 1)[0]: int(row.split(":", 1)[1].split()[0]) for row in rows}
    return round((values["MemTotal"] - values["MemAvailable"]) / 1024, 1)

report = {
    "schema": "kandinsky_persistent_worker_pilot_v1",
    "gpu": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "source_commit": spec["source_commit"],
    "checkpoint": "kandinskylab/Kandinsky-6.0-Lite-5s-Diffusers (Full Lite)",
    "checkpoint_revisions": spec["checkpoint_revisions"],
    "prompt": spec["prompt"],
    "duration_seconds": spec["duration_seconds"],
    "input_video": Path(spec["input_video"]).name,
    "input_video_sha256": spec["input_video_sha256"],
    "config": Path(spec["config"]).name,
    "load_seconds": None,
    "requests": [],
    "error": None
}

def save_report():
    temp = Path(spec["report_path"] + ".tmp")
    temp.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(spec["report_path"])

try:
    torch.cuda.synchronize()
    load_started = time.perf_counter()
    pipe = get_pipeline(spec["config"], device="cuda", attention_engine="sdpa")
    torch.cuda.synchronize()
    report["load_seconds"] = round(time.perf_counter() - load_started, 2)
    report["pipeline_loaded_at_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    print(f"Pipeline load: {report['load_seconds']} seconds", flush=True)
    save_report()

    for seed in spec["seeds"]:
        output_path = output_dir / f"city_walkers_seed_{seed}.mp4"
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        pipe(
            text=spec["prompt"],
            video=spec["input_video"],
            time_length=spec["duration_seconds"],
            seed=seed,
            save_path=str(output_path),
            show_progress=False,
        )
        torch.cuda.synchronize()
        elapsed = round(time.perf_counter() - started, 2)
        if not output_path.is_file() or output_path.stat().st_size == 0:
            raise RuntimeError(f"Output was not written: {output_path}")
        item = {
            "seed": seed,
            "end_to_end_seconds": elapsed,
            "torch_peak_allocated_mib": round(torch.cuda.max_memory_allocated() / (1024**2), 1),
            "torch_peak_reserved_mib": round(torch.cuda.max_memory_reserved() / (1024**2), 1),
            "host_memory_used_mib_after_request": host_memory_mib(),
            "output_file": output_path.name,
            "output_sha256": sha256_file(output_path),
            "output_bytes": output_path.stat().st_size,
        }
        report["requests"].append(item)
        save_report()
        print(f"Seed {seed}: {elapsed} sec | peak allocated {item['torch_peak_allocated_mib']} MiB", flush=True)
except Exception as exc:
    report["error"] = f"{type(exc).__name__}: {exc}"
    report["traceback"] = traceback.format_exc()
    save_report()
    raise
"""
driver_spec = {
    "source_commit": SOURCE_COMMIT,
    "checkpoint_revisions": CHECKPOINT_REVISIONS,
    "prompt": PROMPT,
    "duration_seconds": DURATION_SECONDS,
    "input_video": str(INPUT_VIDEO),
    "input_video_sha256": window_hash,
    "config": str(CONFIG),
    "seeds": [42, 123, 7],
    "output_dir": str(OUTPUT_DIR),
    "report_path": str(REPORT),
}
DRIVER.write_text(driver_source, encoding="utf-8")
spec_json = json.dumps(driver_spec, ensure_ascii=False)
driver_launcher = WORK / "launch_worker.py"
driver_launcher.write_text(
    "import json,runpy,sys\nspec=json.loads(" + repr(spec_json) +
    ")\nsys.argv=[r'" + str(DRIVER) + "', json.dumps(spec)]\nrunpy.run_path(r'" +
    str(DRIVER) + "', run_name='__main__')\n",
    encoding="utf-8"
)

def sample_resources(elapsed):
    gpu = subprocess.run([
        "nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
        "--format=csv,noheader,nounits"
    ], capture_output=True, text=True)
    row = next(csv.reader([gpu.stdout.strip()])) if gpu.returncode == 0 and gpu.stdout.strip() else []
    mem = {x.split(":",1)[0]: int(x.split(":",1)[1].split()[0])
           for x in Path("/proc/meminfo").read_text().splitlines() if ":" in x}
    return {
        "elapsed_seconds": round(elapsed, 1),
        "gpu_name": row[0].strip() if len(row) > 0 else "",
        "gpu_total_mib": int(row[1]) if len(row) > 1 else None,
        "gpu_used_mib": int(row[2]) if len(row) > 2 else None,
        "gpu_utilization_percent": int(row[3]) if len(row) > 3 else None,
        "host_memory_used_mib": round((mem["MemTotal"]-mem["MemAvailable"])/1024, 1),
    }

resource_samples = []
start = time.monotonic()
with open(RUN_LOG, "w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [str(PY), str(driver_launcher)],
        cwd=REPO, env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT, text=True
    )
    while process.poll() is None:
        resource_samples.append(sample_resources(time.monotonic() - start))
        time.sleep(1.0)
    return_code = process.wait()
resource_samples.append(sample_resources(time.monotonic() - start))

with open(RESOURCE_DIR / "worker_resources.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(resource_samples[0].keys()))
    writer.writeheader()
    writer.writerows(resource_samples)

if not REPORT.is_file() or REPORT.read_text(encoding="utf-8").strip() == "{}":
    print(RUN_LOG.read_text(errors="replace")[-7000:])
    raise RuntimeError("Worker exited before writing a report.")
worker_report = json.loads(REPORT.read_text(encoding="utf-8"))
print(RUN_LOG.read_text(errors="replace")[-5000:])
if return_code != 0 or worker_report.get("error"):
    raise RuntimeError("Persistent worker run failed; inspect worker.log and worker_report.json.")
if len(worker_report["requests"]) != 3:
    raise RuntimeError(f"Expected 3 completed jobs, got {len(worker_report['requests'])}.")

warm_times = [x["end_to_end_seconds"] for x in worker_report["requests"][1:]]
print("Model load seconds:", worker_report["load_seconds"])
print("First request seconds:", worker_report["requests"][0]["end_to_end_seconds"])
print("Warm request times:", warm_times, "| warm median:", statistics.median(warm_times))
print("Sampled peak GPU memory:", max((x["gpu_used_mib"] or 0) for x in resource_samples), "MiB")
print("Sampled peak host RAM:", max(x["host_memory_used_mib"] for x in resource_samples), "MiB")


## Step 6 — Listen and export one report package

Listen for scene match, timing, audio quality, and obvious artifacts. The ZIP contains the exact input, all three outputs, environment, settings, resource samples, logs, and hashes. It excludes model weights.


In [ ]:
from google.colab import files

for item in worker_report["requests"]:
    wav_path = OUTPUT_DIR / (Path(item["output_file"]).stem + ".wav")
    extracted = subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(OUTPUT_DIR / item["output_file"]),
        "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(wav_path)
    ], capture_output=True, text=True)
    if extracted.returncode or not wav_path.is_file():
        raise RuntimeError("Could not extract review WAV: " + extracted.stderr[-1500:])
    item["audio_file"] = wav_path.name
    item["audio_sha256"] = sha256_file(wav_path)
    print("Seed", item["seed"], "|", item["output_file"])
    display(Audio(filename=str(wav_path)))

worker_report["listening_notes"] = ""
worker_report["review_ratings_1_to_5"] = {
    "scene_match": None,
    "sync": None,
    "audio_quality": None,
    "artifact_control": None,
}
WORKER_REPORT_PATH = WORK / "worker_report.json"
WORKER_REPORT_PATH.write_text(json.dumps(worker_report, ensure_ascii=False, indent=2), encoding="utf-8")

manifest = {
    "experiment": "kandinsky_persistent_worker_pilot_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "gpu_profile": GPU_PROFILE,
    "gpu_name": GPU_NAME,
    "gpu_memory_mib": GPU_MEMORY_MIB,
    "gpu_compute_capability": GPU_COMPUTE_CAP,
    "source_page": SOURCE_PAGE,
    "source_sha256": source_hash,
    "input_window_sha256": window_hash,
    "expected_previous_window_sha256": EXPECTED_WINDOW_SHA256,
    "model_load_seconds": worker_report["load_seconds"],
    "job_count": len(worker_report["requests"]),
    "purpose": "Measure one loaded Kandinsky pipeline serving three sequential V2A requests; no training.",
}
MANIFEST = WORK / "manifest.json"
MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

BUNDLE = WORK / f"kandinsky_v2a_persistent_worker_{GPU_PROFILE}.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(MANIFEST, "manifest.json")
    archive.write(WORKER_REPORT_PATH, "worker_report.json")
    archive.write(WORK / "environment_packages.txt", "environment_packages.txt")
    archive.write(CONFIG, "aws_persistent_worker_lite.yaml")
    archive.write(INPUT_VIDEO, "input/" + INPUT_VIDEO.name)
    for item in worker_report["requests"]:
        for key in ("output_file", "audio_file"):
            archive.write(OUTPUT_DIR / item[key], "outputs/" + item[key])
    archive.write(RESOURCE_DIR / "worker_resources.csv", "resources/worker_resources.csv")
    archive.write(RUN_LOG, "logs/worker.log")
    archive.write(DRIVER, "worker/persistent_worker_driver.py")
print("Report package:", BUNDLE, "|", round(BUNDLE.stat().st_size / (1024**2), 2), "MiB")
files.download(str(BUNDLE))
